# faultline quickstart

Catch the moment an AI agent **confidently does the wrong thing with no error** — a tool returns bad data and the agent acts on it. Deterministic, no LLM judge. Runs offline in this notebook.

Four steps: wrap a tool → write the rule → break the tool → read the verdict.

In [ ]:
%pip install -q faultline
import faultline as fl

### 1. Your agent + tools
Wrap each tool with `@fl.tool` (and `fl.wrap(..., is_action=True)` for real side-effects, so they're stubbed during testing). This agent has a bug: it always orders 5, no matter the stock.

In [ ]:
@fl.tool
def get_stock(sku):
    return {"sku": sku, "on_hand": 2}

_orders = []
place_order = fl.wrap(lambda sku, qty: _orders.append(qty), is_action=True, name="place_order")

def agent(task):
    on_hand = get_stock(task["sku"])["on_hand"]
    place_order(task["sku"], 5)          # the bug: always orders 5
    return {"ordered": 5, "on_hand": on_hand}

### 2. The rule that must always hold
An invariant returns a message **if violated**, else `None`. Here: never order more than the stock it read.

In [ ]:
def never_oversell(run):
    for ev in run["events"]:
        if ev.get("is_action") and ev["tool"] == "place_order":
            qty = (ev.get("args") or [None, None])[1]
            on_hand = next((e["result"]["on_hand"] for e in run["events"] if e["tool"] == "get_stock"), None)
            if on_hand is not None and qty > on_hand:
                return f"ordered {qty} with only {on_hand} in stock"
    return None

### 3 + 4. Break the tool and read the verdict
`fl.check` breaks `get_stock` (a stale-cache-style wrong number) and re-runs the agent. The agent acts on the corrupted stock with no error of its own — a **silent failure** — and faultline catches it.

In [ ]:
res = fl.check(
    agent,
    {"sku": "A-12"},
    faults=[fl.WrongNumber(targets=["get_stock"])],
    invariants=[never_oversell],
    trials=3,
)
res.report()
print("\nsilent failures caught:", len(res.silent))

That's the whole idea. Next: `fl.scan(agent, task)` needs **no** invariant (zero-config), `fl.assert_resilient(agent, task)` drops into pytest, and `fl.instrument(graph)` wires it into LangGraph / LangChain / LlamaIndex / pydantic-ai / crewAI.

Full docs: [github.com/aaravanmay/faultline](https://github.com/aaravanmay/faultline) · honest scope in `CAPABILITIES.md`.